# 03 - Run Hybrid Models

Trains hybrid quantum-classical students (NoKD and KD).

**Recommended env:** `conda activate hybridqml311` (Python 3.11)

Run in two stages:
1. **Smoke test** (`n_splits=2, epochs=5`) — verify everything works (~2-3 min)
2. **Full run** (`n_splits=5, epochs=60`) — produce paper results (~15-30 min)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = os.path.abspath('../data/raw/wdbc.csv')
assert os.path.exists(CSV), f'Data file not found: {CSV}'
print('CSV ready:', CSV)

## Stage 1 — Smoke Test (run this first)

In [ ]:
# ---- Smoke test parameters ----
SMOKE = True   # set to False for full run

N_SPLITS = 2  if SMOKE else 5
EPOCHS   = 5  if SMOKE else 60
BATCH    = 8  if SMOKE else 16
N_Q_LAYERS = 1
PCA_DIM    = 4
N_QUBITS   = 4
print(f'Mode: {"SMOKE" if SMOKE else "FULL"}  | splits={N_SPLITS}  epochs={EPOCHS}  batch={BATCH}')

## Step 1 — Train Teacher

In [ ]:
teacher = train_teacher_cv(
    CSV, pca_dim=PCA_DIM, n_splits=N_SPLITS, seed=42,
    epochs=40 if SMOKE else 80
)
print('Teacher done.')

## Step 2 — Student Hybrid NoKD

In [ ]:
metrics_nokd = run_student_cv(
    CSV,
    teacher_fold_outputs=None,
    model_type='hybrid',
    use_kd=False,
    quantum_position='middle',
    pca_dim=PCA_DIM,
    n_qubits=N_QUBITS,
    n_q_layers=N_Q_LAYERS,
    n_splits=N_SPLITS,
    seed=42,
    epochs=EPOCHS,
    lr=1e-3,
    batch_size=BATCH,
    exp_name='student_hybrid_nokd'
)
print('Hybrid NoKD done.')

## Step 3 — Student Hybrid KD

In [ ]:
metrics_kd = run_student_cv(
    CSV,
    teacher_fold_outputs=teacher,
    model_type='hybrid',
    use_kd=True,
    alpha=0.5,
    T=2.0,
    quantum_position='middle',
    pca_dim=PCA_DIM,
    n_qubits=N_QUBITS,
    n_q_layers=N_Q_LAYERS,
    n_splits=N_SPLITS,
    seed=42,
    epochs=EPOCHS,
    lr=1e-3,
    batch_size=BATCH,
    exp_name='student_hybrid_kd'
)
print('Hybrid KD done.')

## Results Summary

In [ ]:
import numpy as np

def print_summary(name, metrics):
    print(f'--- {name} ---')
    for k in ['auc', 'f1', 'mcc', 'acc']:
        vals = [m[k] for m in metrics]
        print(f'  {k}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')
    t = [m['train_time'] for m in metrics]
    print(f'  train_time: {np.mean(t):.1f}s/fold')

print_summary('Hybrid-NoKD', metrics_nokd)
print_summary('Hybrid-KD',   metrics_kd)